<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/line_gemini_image_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 有料版のトークンが必要。。。。。。




---



---



---



# LINE × Gemini 画像生成 Bot（Colab動作確認版）

LINE公式アカウントに送ったテキストをそのままプロンプトにして、**Gemini API（Nano Banana）**で画像を生成し、LINEに画像で返信します。

**仕組み**
- LINEテキスト受信 → Geminiで画像生成 → 一時保管 → ngrok経由の公開URLで画像メッセージを返信
- Flask + ngrok 構成（既存のWebhookノートブックを拡張）

**重要ポイント（ここでハマりやすい）**
1. LINEの画像メッセージは画像データを直接送れず、`originalContentUrl` に **公開HTTPS URL（JPEG）** が必要。→ Flaskに `/images/<id>.jpg` を立て、ngrok URLを公開URLとして使う。
2. 画像生成は数秒〜十数秒かかる。Webhookをブロックするとタイムアウト→LINEが再送する。→ **別スレッドで生成・返信し、Webhookには即200を返す**。replyTokenは約1分有効なので間に合う。
3. GeminiはPNGを返すが、LINE画像メッセージは **JPEG** 前提。→ PILでJPEGに変換。

**注意（本番運用には不向き）**
- Colabランタイムは時間で切れる（アイドル90分・最大12時間程度）。
- ngrok無料版は起動のたびにURLが変わる。毎回Webhook URLの貼り直しが必要。
- 無料枠のレート上限（画像生成は概ね 1日あたり数百回程度）に注意。最新値は <https://ai.google.dev/gemini-api/docs/rate-limits>。

セルは上から順に実行してください。

## 0. 事前準備

- **LINE**: Channel secret / Channel access token（LINE Developers）
- **ngrok**: authtoken（<https://dashboard.ngrok.com/get-started/your-authtoken> で無料取得）
- **Gemini**: API キー（<https://aistudio.google.com/apikey>）。
  Colab左の **🔑（シークレット）** に `GOOGLE_API_KEY` で登録して「ノートブックからのアクセス」をオンにすると安全です。

## 1. ライブラリのインストール

In [ ]:
!pip install flask pyngrok google-genai pillow --quiet
print("done")

## 2. 設定値の入力

`GOOGLE_API_KEY` はシークレット登録があればそれを使い、なければ下の文字列を使います。

In [ ]:
CHANNEL_SECRET = "" #"ここにChannel secret" #
CHANNEL_ACCESS_TOKEN = "" #"ここにChannel access token" #
NGROK_AUTHTOKEN = "" #"ここにngrokのauthtoken"

# Gemini APIキー: シークレット優先、なければ手入力値
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = ""#"ここにGemini APIキー"

LINE_REPLY_URL = "https://api.line.me/v2/bot/message/reply"

# 画像生成モデル。無料枠のある Nano Banana を既定に。
#   "gemini-2.5-flash-image"  … Nano Banana（高速・無料枠あり）← 既定
#   "gemini-3.1-flash-image"  … Nano Banana 2（新しい・高品質）
#   "gemini-3-pro-image"      … Nano Banana Pro（最高品質・無料枠なし）
IMAGE_MODEL = "gemini-2.5-flash-image"
print("設定OK")

## 3. Gemini クライアントと画像生成関数

プロンプト（=ユーザーが送った文字列）から画像を生成し、**JPEGバイト列**を返します。
安全フィルタ等で画像が返らなかった場合は `None` を返します。

In [ ]:
from google import genai
from PIL import Image
from io import BytesIO

genai_client = genai.Client(api_key=GOOGLE_API_KEY)


def generate_image_jpeg(prompt_text):
    """プロンプトから画像を生成し、JPEGバイト列を返す。生成できなければNone。"""
    response = genai_client.models.generate_content(
        model=IMAGE_MODEL,
        contents=[prompt_text],
    )
    for part in response.parts:
        if part.inline_data is not None:
            img = part.as_image()              # PIL.Image
            if img.mode != "RGB":              # LINEはJPEG前提なのでRGBへ
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=90)
            return buf.getvalue()
    return None


# 動作確認（任意）: 生成できるか1枚試す
_test = generate_image_jpeg("a cute cat astronaut, simple illustration")
print("生成テスト:", "OK" if _test else "画像が返りませんでした", "/ bytes:", len(_test) if _test else 0)

## 4. 署名検証・返信・画像保管の関数

署名検証は元のCGIコードと同じHMAC-SHA256です。
返信はテキスト用と画像用を用意。生成画像はメモリ上に一時保管し、`/images/<id>.jpg` で配信します。

In [ ]:
import hmac, hashlib, base64, json, uuid, time
import urllib.request

# 生成画像の一時保管(メモリ上)。{id: (jpeg_bytes, 作成時刻)}
IMAGE_STORE = {}
STORE_TTL = 600  # 10分で自動掃除


def verify_signature(body_bytes, signature):
    """LINE署名検証。Channel secretでHMAC-SHA256を計算し一致を確認。生のバイト列を使うのが重要。"""
    if not CHANNEL_SECRET or CHANNEL_SECRET == "ここにChannel secret":
        return True  # 未設定時はスキップ(動作確認用。本番では危険)
    digest = hmac.new(CHANNEL_SECRET.encode("utf-8"), body_bytes, hashlib.sha256).digest()
    expected = base64.b64encode(digest).decode("utf-8")
    return hmac.compare_digest(expected, signature)


def put_image(jpeg_bytes):
    """画像を保管してIDを返す。ついでに古い画像を掃除。"""
    img_id = uuid.uuid4().hex
    IMAGE_STORE[img_id] = (jpeg_bytes, time.time())
    now = time.time()
    for k in list(IMAGE_STORE):
        if now - IMAGE_STORE[k][1] > STORE_TTL:
            IMAGE_STORE.pop(k, None)
    return img_id


def _post_line_reply(data):
    req = urllib.request.Request(
        LINE_REPLY_URL,
        data=json.dumps(data).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {CHANNEL_ACCESS_TOKEN}",
        },
        method="POST",
    )
    urllib.request.urlopen(req)


def reply_text(reply_token, text):
    _post_line_reply({
        "replyToken": reply_token,
        "messages": [{"type": "text", "text": text}],
    })


def reply_image(reply_token, image_url):
    _post_line_reply({
        "replyToken": reply_token,
        "messages": [{
            "type": "image",
            "originalContentUrl": image_url,   # 公開HTTPS・JPEG
            "previewImageUrl": image_url,
        }],
    })

print("関数を定義しました")

## 5. Flaskアプリの定義

`/callback` … LINEからのWebhook。**画像生成は別スレッド**に投げ、LINEには即200を返します。
`/images/<id>.jpg` … 生成画像の配信（LINEサーバーがここを取りに来る）。

In [ ]:
from flask import Flask, request, Response
import threading

app = Flask(__name__)
PUBLIC_URL = None  # ngrok起動後にセットされる


def handle_event_async(reply_token, user_text):
    """別スレッドで実行: 画像生成 → 返信。"""
    try:
        jpeg = generate_image_jpeg(user_text)
        if jpeg is None:
            reply_text(reply_token, "画像を生成できませんでした。別の表現で試してください。")
            return
        img_id = put_image(jpeg)
        image_url = f"{PUBLIC_URL}/images/{img_id}.jpg"
        reply_image(reply_token, image_url)
    except Exception as e:
        print("生成/返信エラー:", e)
        try:
            reply_text(reply_token, f"エラーが発生しました: {e}")
        except Exception:
            pass


@app.route("/callback", methods=["POST"])
def callback():
    body_bytes = request.get_data()                       # 生バイト(署名検証用)
    signature = request.headers.get("X-Line-Signature", "")
    if not verify_signature(body_bytes, signature):
        return "NG", 400

    data = json.loads(body_bytes.decode("utf-8"))
    for event in data.get("events", []):
        if event.get("type") != "message":
            continue
        message = event.get("message", {})
        if message.get("type") != "text":
            continue
        reply_token = event.get("replyToken")
        user_text = message.get("text", "")
        # 生成は時間がかかるので別スレッドへ。LINEには即200を返す(再送防止)。
        threading.Thread(
            target=handle_event_async,
            args=(reply_token, user_text),
            daemon=True,
        ).start()

    return "OK", 200


@app.route("/images/<img_id>.jpg", methods=["GET"])
def serve_image(img_id):
    item = IMAGE_STORE.get(img_id)
    if item is None:
        return "Not found", 404
    return Response(item[0], mimetype="image/jpeg")


@app.route("/", methods=["GET"])
def index():
    return "LINE x Gemini image bot is running."

print("Flaskアプリを定義しました")

## 6. ngrokでトンネルを開いてサーバー起動

実行すると `Webhook URL: https://xxxx.ngrok-free.app/callback` が表示されます。
これを LINE Developers の **Webhook URL** に貼り付け、「検証」を押してください。

> このセルは起動したままになります。停止するとサーバーも止まります。

In [ ]:
from pyngrok import ngrok, conf

# 既存トンネルを掃除
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

conf.get_default().auth_token = NGROK_AUTHTOKEN

PORT = 5000
PUBLIC_URL = ngrok.connect(PORT).public_url   # ← グローバルに反映(画像URLに使う)
print("=" * 55)
print(f"Webhook URL: {PUBLIC_URL}/callback")
print("↑ これを LINE Developers の Webhook URL に設定")
print("=" * 55)

def run():
    app.run(port=PORT, use_reloader=False)

threading.Thread(target=run, daemon=True).start()
print("サーバー起動中。LINEからテキストを送ると画像が返ります。")

## 補足：動作確認の手順

1. LINE Developers → Messaging API設定 → Webhook URL に上記URLを設定し「検証」
2. 「Webhookの利用」をオンにする
3. 自動応答メッセージは必要に応じてオフ
4. ボットを友だち追加して、トークで「赤い帽子をかぶった柴犬」などと送信
5. 数秒〜十数秒で画像が返ってくれば成功

### うまくいかないとき
- **テキストは返るが画像が表示されない**: まずブラウザで `https://xxxx.ngrok-free.app/images/...` を直接開いて画像が出るか確認。ngrok無料版が警告ページ（interstitial）を挟んでLINEが画像を取得できていない場合があります。その時は ngrok のagent設定で `request_header.add: ["ngrok-skip-browser-warning:true"]` を付ける、または有料の固定ドメインを使うと確実です。
- **画像が生成されない（テキストで「生成できませんでした」）**: プロンプトが安全フィルタに当たっている可能性。表現を変える。
- **429 / RESOURCE_EXHAUSTED**: 無料枠のレート上限。少し待つ、`IMAGE_MODEL` を見直す。
- **PERMISSION_DENIED**: Gemini APIキーが未設定/誤り。シークレット名 `GOOGLE_API_KEY` を確認。
- **返信が来ない**: セル5の `生成/返信エラー` ログと、ngrok管理画面のリクエストログを確認。

### 発展
- 「猫」だけ送られても良い画像になるよう、生成前にプロンプトを補強できます（例: `f"{user_text}、高品質なイラスト、白背景"`）。
- テキスト応答も混ぜたい場合は、`message.text` が「画像:」で始まったら画像、そうでなければ通常のGemini文章応答、と分岐させると一つのBotで両対応できます。